# 2474. Customers With Strictly Increasing Purchases

## Problem Description
We need to find customers whose **total purchases are strictly increasing year by year**.  

- The total purchases of a customer in one year = sum of `price` for all orders in that year.  
- If a customer has no orders in a year, their total purchases for that year = 0.  
- The first year to consider = year of their first order.  
- The last year to consider = year of their last order.  
- Return the IDs of customers whose yearly totals are strictly increasing.  

---

## Schema

### Table: Orders
| Column Name  | Type | Description                                      |
|--------------|------|--------------------------------------------------|
| order_id     | INT  | Primary key, unique order identifier             |
| customer_id  | INT  | ID of the customer who placed the order          |
| order_date   | DATE | Date of the order                                |
| price        | INT  | Price of the order                               |

---

## Sample Data

### Orders
| order_id | customer_id | order_date | price |
|----------|-------------|------------|-------|
| 1        | 1           | 2019-07-01 | 1100  |
| 2        | 1           | 2019-11-01 | 1200  |
| 3        | 1           | 2020-05-26 | 3000  |
| 4        | 1           | 2021-08-31 | 3100  |
| 5        | 1           | 2022-12-07 | 4700  |
| 6        | 2           | 2015-01-01 | 700   |
| 7        | 2           | 2017-11-07 | 1000  |
| 8        | 3           | 2017-01-01 | 900   |
| 9        | 3           | 2018-11-07 | 900   |

---

## Expected Output
| customer_id |
|-------------|
| 1           |

### Explanation
- **Customer 1:**  
  - 2019 → 2300  
  - 2020 → 3000  
  - 2021 → 3100  
  - 2022 → 4700  
  → Strictly increasing, included.  

- **Customer 2:**  
  - 2015 → 700  
  - 2016 → 0  
  - 2017 → 1000  
  → Not strictly increasing (0 breaks the sequence), excluded.  

- **Customer 3:**  
  - 2017 → 900  
  - 2018 → 900  
  → Not strictly increasing (equal values), excluded.  

---

## PySpark Code: Create DataFrame and Temp View

```python
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
from datetime import date

# Schema for Orders
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("price", IntegerType(), False)
])

# Data for Orders
orders_data = [
    (1, 1, date(2019,7,1), 1100),
    (2, 1, date(2019,11,1), 1200),
    (3, 1, date(2020,5,26), 3000),
    (4, 1, date(2021,8,31), 3100),
    (5, 1, date(2022,12,7), 4700),
    (6, 2, date(2015,1,1), 700),
    (7, 2, date(2017,11,7), 1000),
    (8, 3, date(2017,1,1), 900),
    (9, 3, date(2018,11,7), 900)
]

# Create DataFrame
orders_df = spark.createDataFrame(orders_data, orders_schema)

# Register Temp View
orders_df.createOrReplaceTempView("Orders")

# Quick check
orders_df.show()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
from datetime import date

# Schema for Orders
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("price", IntegerType(), False)
])

# Data for Orders
orders_data = [
    (1, 1, date(2019,7,1), 1100),
    (2, 1, date(2019,11,1), 1200),
    (3, 1, date(2020,5,26), 3000),
    (4, 1, date(2021,8,31), 3100),
    (5, 1, date(2022,12,7), 4700),
    (6, 2, date(2015,1,1), 700),
    (7, 2, date(2017,11,7), 1000),
    (8, 3, date(2017,1,1), 900),
    (9, 3, date(2018,11,7), 900)
]

# Create DataFrame
orders_df = spark.createDataFrame(orders_data, orders_schema)

# Register Temp View
orders_df.createOrReplaceTempView("Orders")

# Quick check
orders_df.show()



In [0]:
%sql
WITH cte AS (
		SELECT DISTINCT sum(price) OVER (
				PARTITION BY customer_id,
				year(order_date)
				) AS total_price,
			year(order_date) AS order_year,
			customer_id
		FROM Orders
		),
	cte2(SELECT (
			total_price - coalesce(lag(total_price, 1) OVER (
					PARTITION BY customer_id ORDER BY order_year ASC
					), 0)
			) AS prev_trans_diff, (
			order_year - coalesce(lag(order_year, 1) OVER (
					PARTITION BY customer_id ORDER BY order_year ASC
					), 0)
			) AS prev_year_diff, row_number() OVER (
			PARTITION BY customer_id ORDER BY order_year ASC
			) AS rn, * FROM cte ORDER BY customer_id ASC, order_year ASC),
	cte3 AS (
		SELECT CASE 
				WHEN rn <> 1
					AND prev_year_diff <> 1
					THEN NULL
				WHEN rn = 1
					THEN 1
				WHEN rn <> 1
					AND prev_year_diff = 1
					AND prev_trans_diff > 0
					THEN 1
				END AS STATUS,
			customer_id
		FROM cte2
		)

SELECT DISTINCT customer_id
FROM cte3
WHERE customer_id NOT IN (
		SELECT DISTINCT customer_id
		FROM cte3
		WHERE STATUS IS NULL
		)



# Documentation: Customers With Strictly Increasing Purchases

This query determines which customers have **strictly increasing yearly purchases** between their first and last order year.
```markdown


---

## Step 1: Calculate Yearly Totals (CTE)
```sql
WITH cte AS (
    SELECT DISTINCT 
           SUM(price) OVER (
               PARTITION BY customer_id, YEAR(order_date)
           ) AS total_price,
           YEAR(order_date) AS order_year,
           customer_id
    FROM Orders
)
```
- For each `customer_id` and `order_year`, compute the **total purchases** using `SUM(price)` with a window function.  
- `DISTINCT` ensures we don’t duplicate rows.  
- Output columns:  
  - `total_price` → yearly total purchases.  
  - `order_year` → year of the order.  
  - `customer_id` → customer identifier.

---

## Step 2: Compare With Previous Year (CTE2)
```sql
cte2 AS (
    SELECT 
        (total_price - COALESCE(
            LAG(total_price, 1) OVER (
                PARTITION BY customer_id ORDER BY order_year ASC
            ), 0)
        ) AS prev_trans_diff,
        (order_year - COALESCE(
            LAG(order_year, 1) OVER (
                PARTITION BY customer_id ORDER BY order_year ASC
            ), 0)
        ) AS prev_year_diff,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id ORDER BY order_year ASC
        ) AS rn,
        *
    FROM cte
    ORDER BY customer_id ASC, order_year ASC
)
```
- For each customer, compare the current year’s totals with the **previous year**:
  - `prev_trans_diff` → difference in total purchases compared to the previous year.  
  - `prev_year_diff` → difference in year compared to the previous year.  
- `ROW_NUMBER()` assigns sequential order to each year per customer.  
- This step prepares the data to check if purchases are strictly increasing year by year.

---

## Step 3: Determine Valid Status (CTE3)
```sql
cte3 AS (
    SELECT CASE 
               WHEN rn <> 1 AND prev_year_diff <> 1 THEN NULL
               WHEN rn = 1 THEN 1
               WHEN rn <> 1 AND prev_year_diff = 1 AND prev_trans_diff > 0 THEN 1
           END AS STATUS,
           customer_id
    FROM cte2
)
```
- Apply rules to decide if a customer’s yearly sequence is valid:
  - **First year (`rn = 1`)** → always marked as valid (`STATUS = 1`).  
  - **Non‑first year, but missing consecutive year (`prev_year_diff <> 1`)** → invalid (`STATUS = NULL`).  
  - **Consecutive year with strictly higher purchases (`prev_trans_diff > 0`)** → valid (`STATUS = 1`).  
- This flags each year as either valid or invalid for strictly increasing purchases.

---

## Step 4: Final Selection
```sql
SELECT DISTINCT customer_id
FROM cte3
WHERE customer_id NOT IN (
    SELECT DISTINCT customer_id
    FROM cte3
    WHERE STATUS IS NULL
)
```
- Exclude customers who have **any invalid year** (`STATUS IS NULL`).  
- Return only those customers whose purchases are strictly increasing across all years.  
- `DISTINCT` ensures each customer appears once.

---

## Final Output
- The query returns the list of `customer_id`s whose yearly totals are **strictly increasing** from their first to last order year.  
- Customers with missing years or non‑increasing totals are excluded.

---

## Key Insights
1. **Window functions (`SUM`, `LAG`, `ROW_NUMBER`)** are used to calculate yearly totals and compare them across years.  
2. **CASE logic** determines whether each year’s purchases meet the strictly increasing condition.  
3. The final filter removes customers with any invalid year.  
4. The query ensures only customers with strictly increasing yearly totals are returned.
```

---

